# Principal Component Analysis -- sklearn Digits

This notebook applies PCA to the sklearn handwritten-digits dataset (1,797 samples, 64 pixel
features, 10 digit classes) to demonstrate dimensionality reduction from first principles.

- **Explained variance:** quantify how much information each principal component captures
- **2-D projection:** visualise the 64-dimensional digit space compressed to 2 dimensions
- **Image reconstruction:** show how reconstruction quality changes as fewer components are used
- **Eigenfaces:** display the principal components as 8x8 images to interpret what each axis encodes
- All dimensionality reduction is performed with `mlpackage.PCA`; no sklearn PCA is used

## Mathematical Intuition

### Covariance Matrix

Given a mean-centred data matrix $X \in \mathbb{R}^{n \times d}$, the covariance matrix is:

$$C = \frac{1}{n} X^\top X \in \mathbb{R}^{d \times d}$$

### Eigendecomposition

PCA finds the orthonormal eigenvectors $\mathbf{v}_1, \ldots, \mathbf{v}_d$ of $C$ ordered by
decreasing eigenvalue $\lambda_1 \geq \lambda_2 \geq \cdots \geq \lambda_d \geq 0$.
Each eigenvalue equals the variance of the data projected onto the corresponding eigenvector.
In practice the decomposition is computed via SVD: $X = U \Sigma V^\top$, where the columns of
$V$ are the principal directions.

### Projection

Keeping $k$ components, the low-rank projection is:

$$Z = X V_k \quad (n \times k)$$

where $V_k \in \mathbb{R}^{d \times k}$ holds the top-$k$ eigenvectors as columns.

### Reconstruction and Reconstruction Error

Approximate reconstruction back to the original space:

$$\hat{X} = Z V_k^\top + \bar{\mathbf{x}}$$

The mean squared reconstruction error equals $\sum_{j=k+1}^{d} \lambda_j$ -- the sum of the
discarded eigenvalues -- so using more components always reduces error.

## Dataset Overview

**Source:** sklearn `load_digits` | **Rows:** 1,797 | **Features:** 64 (8x8 pixel intensities, 0-16) | **Target:** digit class 0-9

| Feature | Type | Description |
|---|---|---|
| pixel_0_0 to pixel_7_7 | Continuous | Grayscale pixel intensity in an 8x8 grid (range 0-16) |
| target | Categorical | Handwritten digit class (0 through 9) |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

from sklearn.datasets import load_digits
from mlpackage import PCA, StandardScaler, train_test_split

data = load_digits()
X_raw, y_raw = data.data, data.target

print(f"Shape: {X_raw.shape}")
print(f"Classes: {np.unique(y_raw)}")
print(f"Pixel range: [{X_raw.min()}, {X_raw.max()}]")

## Exploratory Data Analysis

In [ ]:
counts = np.bincount(y_raw)
plt.figure(figsize=(8, 3))
plt.bar(np.arange(10), counts, color="steelblue")
plt.title("Digit Class Distribution")
plt.xlabel("Digit")
plt.ylabel("Count")
plt.xticks(np.arange(10))
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for digit in range(10):
    idx = np.where(y_raw == digit)[0][0]
    ax  = axes[digit // 5][digit % 5]
    ax.imshow(X_raw[idx].reshape(8, 8), cmap="gray_r")
    ax.set_title(f"Digit {digit}")
    ax.axis("off")
plt.suptitle("One Example per Digit Class")
plt.tight_layout()
plt.show()

## Preprocessing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42)

scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {X_train_s.shape}  |  Test: {X_test_s.shape}")

## Fitting PCA

In [ ]:
pca_full = PCA(n_components=64)
pca_full.fit(X_train_s)

evr        = pca_full.explained_variance_ratio_
cumulative = np.cumsum(evr)

print(f"Components to explain 80%: {np.searchsorted(cumulative, 0.80) + 1}")
print(f"Components to explain 90%: {np.searchsorted(cumulative, 0.90) + 1}")
print(f"Components to explain 95%: {np.searchsorted(cumulative, 0.95) + 1}")

## Visualizations

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))

ax1.bar(np.arange(1, 65), evr, color="steelblue", alpha=0.7, label="Individual")
ax1.set_xlabel("Principal Component")
ax1.set_ylabel("Explained Variance Ratio", color="steelblue")
ax1.tick_params(axis="y", labelcolor="steelblue")

ax2 = ax1.twinx()
ax2.plot(np.arange(1, 65), cumulative, color="coral", linewidth=2, label="Cumulative")
ax2.axhline(0.80, color="green",  linestyle="--", linewidth=1, label="80%")
ax2.axhline(0.90, color="orange", linestyle="--", linewidth=1, label="90%")
ax2.axhline(0.95, color="red",    linestyle="--", linewidth=1, label="95%")
ax2.set_ylabel("Cumulative Explained Variance", color="coral")
ax2.tick_params(axis="y", labelcolor="coral")
ax2.set_ylim(0, 1.05)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="center right")
plt.title("Explained Variance per Principal Component")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, 21), evr[:20], marker="o", color="steelblue", linewidth=2)
plt.title("Scree Plot: First 20 Principal Components")
plt.xlabel("Component Index")
plt.ylabel("Explained Variance Ratio")
plt.tight_layout()
plt.show()

In [ ]:
pca_2d = PCA(n_components=2)
pca_2d.fit(X_train_s)
Z_train = pca_2d.transform(X_train_s)

cmap = plt.get_cmap("tab10")
plt.figure(figsize=(9, 7))
for digit in range(10):
    mask = y_train == digit
    plt.scatter(Z_train[mask, 0], Z_train[mask, 1],
                color=cmap(digit), s=15, alpha=0.6, label=str(digit))
plt.title("2-D PCA Projection of Digits Dataset (Train Set)")
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.legend(title="Digit", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
n_components_list = [2, 10, 20, 40, 64]

examples = []
for digit in range(10):
    idx = np.where(y_test == digit)[0][0]
    examples.append(X_test_s[idx])
examples = np.array(examples)

fig, axes = plt.subplots(10, 5, figsize=(10, 20))
for col_idx, k in enumerate(n_components_list):
    pca_k = PCA(n_components=k)
    pca_k.fit(X_train_s)
    Z_ex  = pca_k.transform(examples)
    X_rec = pca_k.inverse_transform(Z_ex)
    for row_idx in range(10):
        ax = axes[row_idx][col_idx]
        ax.imshow(X_rec[row_idx].reshape(8, 8), cmap="gray_r")
        ax.axis("off")
        if row_idx == 0:
            ax.set_title(f"{k} PC", fontsize=10)
        if col_idx == 0:
            ax.set_ylabel(f"Digit {row_idx}", fontsize=9)
plt.suptitle("Digit Reconstruction Quality vs Number of Principal Components", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
components = pca_full.components_

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(components[i].reshape(8, 8), cmap="RdBu_r")
    ax.set_title(f"PC {i+1}", fontsize=9)
    ax.axis("off")
plt.suptitle("First 16 Principal Components (Eigenfaces)")
plt.tight_layout()
plt.show()

## Interpretation and Conclusions

- **The first few principal components capture a disproportionate share of variance:** roughly
  20 components explain approximately 90% of the total variance, meaning the intrinsic
  dimensionality of the digit space is far lower than the 64 raw pixel dimensions.
- **The 2-D scatter shows meaningful cluster separation even in just two dimensions:** digits like
  0 and 1 are well separated, while visually similar digits (3, 5, 8) overlap, confirming that
  PCA captures global structure but not all discriminative detail.
- **Reconstruction quality improves rapidly with more components:** with only 2 components the
  digit outlines are barely recognisable, but at 20 components the shapes are clear, and at 40
  the images are nearly indistinguishable from the originals.
- **The eigenfaces reveal what each principal axis encodes:** early components capture broad
  contrast patterns (dark vs bright regions), while later components encode finer, digit-specific
  stroke features.
- **PCA is unsupervised:** it maximises variance without knowledge of digit labels, yet the
  projections still show class-meaningful geometry, a strong indicator that digit class identity
  is the dominant source of variance in the raw pixel data.